# DAS fin-whale localization — EDA

**Task 3 of DCLDE 2027: detect and localize fin-whale calls on a Distributed Acoustic Sensing (DAS)
array — a fiber-optic cable turned into ~tens-of-thousands of strain sensors.**

`dclde_2027_das-finwhale-localization/` — two cables off Oregon (North & South), 5 batches:
- `detections/batch_*/*.nc` — the **data**: denoised SNR **strain images** (channel × time), two bands.
- `annotations/batch*/annotated_calls_*.csv` — labeled calls: `time`, `dist` (metres along cable),
  `call_type` (LF/HF), `cable`, `snr`.
- `geometry/*_DAS_multicoord.csv` — per-channel lat/lon/depth/UTM (maps channel index ↔ real position).

This is effectively a **computer-vision problem on strain images**, not per-hydrophone waveforms.

In [ ]:
import os, sys, csv, glob
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, REPO)
# Sample cache: fast NVMe scratch on this box if present, else a gitignored repo dir.
_cache = "/opt/dlami/nvme/dclde_samples" if os.path.isdir("/opt/dlami/nvme") else os.path.join(REPO, "data", "_samples")
os.environ.setdefault("DCLDE_SAMPLE_DIR", _cache)
from scripts.fetch_sample import fetch   # downloads single objects from the public bucket on demand

import numpy as np, pandas as pd
import soundfile as sf
from scipy import signal
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print("sample cache:", os.environ["DCLDE_SAMPLE_DIR"])

In [ ]:
import xarray as xr

## 1. The array is a cable — geometry
Load the channel geometry for both cables and plot their real-world track. Channels are ~2 m apart over
tens of km, so the "array" is a dense line on the seafloor. Distance-along-cable is the coordinate the
annotations use.

In [ ]:
geo = {c: pd.read_csv(fetch(f"dclde_2027_das-finwhale-localization/geometry/{c}_DAS_multicoord.csv"))
       for c in ("north", "south")}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for c, g in geo.items():
    axes[0].plot(g["lon"], g["lat"], lw=2, label=f"{c} ({len(g):,} ch)")
    axes[1].plot(g["chan_m"] / 1000, g["depth"], lw=1.5, label=c)
axes[0].set_xlabel("lon"); axes[0].set_ylabel("lat"); axes[0].set_title("cable tracks off Oregon"); axes[0].legend()
axes[1].set_xlabel("distance along cable (km)"); axes[1].set_ylabel("depth (m)"); axes[1].set_title("depth profile"); axes[1].legend()
axes[1].invert_yaxis(); plt.tight_layout(); plt.show()

## 2. The data is a strain image (channels × time), two bands
Open one 70-second `.nc` segment. Dimensions are `dist` (channels) × `time`; variables are `SNR_lf` and
`SNR_hf` (low- and high-frequency band SNR — matching the LF/HF call types). Attributes give the acquisition
physics: sampling rate `fs`, channel spacing `dx`, gauge length `GL`.

In [ ]:
# note: annotation `filename` uses colons (02:05:52); bucket objects use dashes (02-05-52)
nc = fetch("dclde_2027_das-finwhale-localization/detections/batch_1/Denoised_SNR_North_2021-11-04_02-05-52.nc")
ds = xr.open_dataset(nc)
print("dims:", dict(ds.sizes))
print("vars:", list(ds.data_vars))
print({k: round(float(ds.attrs[k]), 3) for k in ("fs", "dx", "GL", "scale_factor") if k in ds.attrs}, "cable:", ds.attrs.get("cablename"))

## 3. Annotated calls trace moveout curves
Plot the strain image (distance × time) and overlay the annotated calls for this segment. Each fin-whale
call sweeps across channels, so it draws a **V-shaped moveout curve** — the apex is the whale's closest
point of approach to the cable. Two cables let you triangulate the source; this V-geometry is the signal
a detector/localizer must learn.

In [ ]:
calls = pd.read_csv(fetch("dclde_2027_das-finwhale-localization/annotations/batch1/annotated_calls_north_2021-11-04_02_05_52.csv"))
print(f"{len(calls)} calls in this 70 s segment:", dict(calls["call_type"].value_counts()))

t = ds["time"].values; d = ds["dist"].values / 1000.0
sd = max(1, ds.sizes["dist"] // 1500); st = max(1, ds.sizes["time"] // 1500)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (band, ctype) in zip(axes, [("SNR_lf", "LF"), ("SNR_hf", "HF")]):
    img = ds[band].values[::sd, ::st]
    ax.imshow(img, aspect="auto", origin="lower", cmap="magma", vmin=0, vmax=np.percentile(img, 99.5),
              extent=[t[0], t[-1], d[0], d[-1]])
    sub = calls[calls["call_type"] == ctype]
    ax.scatter(sub["time"], sub["dist"] / 1000, s=16, facecolors="none", edgecolors="cyan", lw=1.0)
    ax.set_xlabel("time in segment (s)"); ax.set_title(f"{band} — {len(sub)} {ctype} calls")
axes[0].set_ylabel("distance along cable (km)"); plt.tight_layout(); plt.show()

## 4. Call catalog — distributions
Load the annotated calls across all batches and look at how they distribute over distance, SNR, band and
cable. This is the clean, curated label set (unlike NARW's raw candidates).

In [ ]:
import urllib.request, json, urllib.parse
def list_annotation_csvs():
    """List every annotated_calls_*.csv under the DAS annotations/ prefix (public GCS JSON API)."""
    base = "https://storage.googleapis.com/storage/v1/b/noaa-passive-bioacoustic/o"
    prefix = "dclde/2027/dclde_2027_das-finwhale-localization/annotations/"
    token, out = None, []
    while True:
        u = f"{base}?prefix={urllib.parse.quote(prefix)}&maxResults=1000" + (f"&pageToken={token}" if token else "")
        data = json.load(urllib.request.urlopen(u))
        out += [it["name"] for it in data.get("items", []) if it["name"].endswith(".csv")]
        token = data.get("nextPageToken")
        if not token:
            break
    return out

objs = list_annotation_csvs()
calls_all = pd.concat([pd.read_csv(fetch(o[len("dclde/2027/"):])) for o in objs], ignore_index=True)
print(f"{len(calls_all):,} annotated calls across {len(objs)} segment files")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
sns.histplot(calls_all["dist"] / 1000, bins=30, ax=axes[0]); axes[0].set_xlabel("distance along cable (km)")
sns.histplot(calls_all["snr"], bins=30, ax=axes[1]); axes[1].set_xlabel("SNR")
cc = calls_all.groupby(["cable", "call_type"]).size().reset_index(name="n")
sns.barplot(data=cc, x="cable", y="n", hue="call_type", ax=axes[2]); axes[2].set_ylabel("calls")
plt.tight_layout(); plt.show()

## Takeaways
- **Modality:** DAS strain **images** (channel × time), two bands, fs≈200 Hz, ~2 m channels over tens of km.
  Treat detection/localization as computer vision (moveout-curve detection), not waveform PAM.
- **Rich, clean labels:** `time` + `dist` + `snr` per call; two cables enable triangulation
  (~100 m accuracy in the literature).
- **Watch the file-name colon↔dash mapping** between annotations and `.nc` objects.
- **Data volume:** each `.nc` is ~0.5 GB; stream per-segment rather than bulk-downloading (103 GB total).